In [1]:
# Import the required libraries and load the CSV files

from pathlib import Path
import pandas as pd

# Folder where the CSV files are stored
data_dir = Path(
    r"C:\Users\manoj\OneDrive\Documents\logistics_operations\raw data"
)

# Get all CSV files from the folder
csv_files = sorted(data_dir.rglob("*.csv"))

# Read each CSV file and store it in a dictionary
csv_data = {}

for file in csv_files:
    table_name = file.stem
    csv_data[table_name] = pd.read_csv(file, low_memory=False)

print("Number of CSV files:", len(csv_data))

# Display the table names
for table_name in csv_data:
    print(table_name)

Number of CSV files: 14
customers
delivery_events
driver_monthly_metrics
drivers
facilities
fuel_purchases
loads
maintenance_records
routes
safety_incidents
trailers
trips
truck_utilization_metrics
trucks


In [2]:
# Basic overview of all tables
table_overview = []

for table_name, df in csv_data.items():

    # Count the unique values in each column
    total_unique = 0

    for column in df.columns:
        total_unique = total_unique + df[column].nunique()

    # Find the average number of unique values per column
    if len(df.columns) > 0:
        avg_cardinality = round(total_unique / len(df.columns), 1)
    else:
        avg_cardinality = 0

    # Store the table details
    table_overview.append({
        "table_name": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_values": df.duplicated().sum(),
        "total_missing_values": df.isna().sum().sum(),
        "total_unique_values": total_unique,
        "avg_cardinality_per_col": avg_cardinality
    })

table_df = pd.DataFrame(table_overview)
table_df = table_df.sort_values("table_name")

display(table_df)

,table_name,rows,columns,duplicate_values,total_missing_values,total_unique_values,avg_cardinality_per_col
0,customers,200,8,0,0,695,86.9
1,delivery_events,170820,11,0,0,612417,55674.3
2,driver_monthly_metrics,4464,9,0,0,13291,1476.8
3,drivers,150,12,0,124,736,61.3
4,facilities,50,9,0,0,216,24.0
5,fuel_purchases,196442,11,0,7868,544086,49462.4
6,loads,85410,12,0,0,197918,16493.2
7,maintenance_records,2920,12,0,0,16229,1352.4
8,routes,58,9,0,0,253,28.1
9,safety_incidents,170,15,0,2,1052,70.1


In [3]:
# Check the data type of each column

for table_name, df in csv_data.items():

    print("\nTable:", table_name)

    column_types = df.dtypes.reset_index()
    column_types.columns = ["column_name", "data_type"]

    display(column_types)


Table: customers


,column_name,data_type
0,customer_id,str
1,customer_name,str
2,customer_type,str
3,credit_terms_days,int64
4,primary_freight_type,str
5,account_status,str
6,contract_start_date,str
7,annual_revenue_potential,int64



Table: delivery_events


,column_name,data_type
0,event_id,str
1,load_id,str
2,trip_id,str
3,event_type,str
4,facility_id,str
5,scheduled_datetime,str
6,actual_datetime,str
7,detention_minutes,int64
8,on_time_flag,bool
9,location_city,str



Table: driver_monthly_metrics


,column_name,data_type
0,driver_id,str
1,month,str
2,trips_completed,int64
3,total_miles,int64
4,total_revenue,float64
5,average_mpg,float64
6,total_fuel_gallons,float64
7,on_time_delivery_rate,float64
8,average_idle_hours,float64


... (18 outputs left out)


Table: truck_utilization_metrics


,column_name,data_type
0,truck_id,str
1,month,str
2,trips_completed,int64
3,total_miles,int64
4,total_revenue,float64
5,average_mpg,float64
6,maintenance_events,int64
7,maintenance_cost,float64
8,downtime_hours,float64
9,utilization_rate,float64



Table: trucks


,column_name,data_type
0,truck_id,str
1,unit_number,int64
2,make,str
3,model_year,int64
4,vin,str
5,acquisition_date,str
6,acquisition_mileage,int64
7,fuel_type,str
8,tank_capacity_gallons,int64
9,status,str


In [4]:
# Validate the primary keys

# List the primary key column for each table
primary_keys = {
    "customers": ["customer_id"],
    "delivery_events": ["event_id"],
    "driver_monthly_metrics": None,
    "drivers": ["driver_id"],
    "facilities": ["facility_id"],
    "fuel_purchases": ["fuel_purchase_id"],
    "loads": ["load_id"],
    "maintenance_records": ["maintenance_id"],
    "routes": ["route_id"],
    "safety_incidents": ["incident_id"],
    "trailers": ["trailer_id"],
    "trips": ["trip_id"],
    "truck_utilization_metrics": None,
    "trucks": ["truck_id"]
}

pk = []

for table_name, key_columns in primary_keys.items():

    # Skip tables that use a composite key
    if key_columns is None:
        continue

    df = csv_data[table_name]

    for column in key_columns:

        pk.append({
            "table": table_name,
            "primary_key": column,
            "total_rows": len(df),
            "duplicate_keys": df[column].duplicated().sum(),
            "missing_keys": df[column].isna().sum(),
            "unique_keys": df[column].nunique()
        })

pk_df = pd.DataFrame(pk)

display(pk_df)

,table,primary_key,total_rows,duplicate_keys,missing_keys,unique_keys
0,customers,customer_id,200,0,0,200
1,delivery_events,event_id,170820,0,0,170820
2,drivers,driver_id,150,0,0,150
3,facilities,facility_id,50,0,0,50
4,fuel_purchases,fuel_purchase_id,196442,0,0,196442
5,loads,load_id,85410,0,0,85410
6,maintenance_records,maintenance_id,2920,0,0,2920
7,routes,route_id,58,0,0,58
8,safety_incidents,incident_id,170,0,0,170
9,trailers,trailer_id,180,0,0,180


In [5]:
# Validate the composite keys

# These tables use more than one column to identify a record
composite_keys = {
    "driver_monthly_metrics": ["driver_id", "month"],
    "truck_utilization_metrics": ["truck_id", "month"]
}

ck = []

for table_name, key_columns in composite_keys.items():

    df = csv_data[table_name]

    duplicate_count = df.duplicated(subset=key_columns).sum()
    missing_count = df[key_columns].isna().sum().sum()
    unique_count = df[key_columns].drop_duplicates().shape[0]

    ck.append({
        "table": table_name,
        "key_columns": " + ".join(key_columns),
        "duplicate_combinations": duplicate_count,
        "missing_key_values": missing_count,
        "unique_combinations": unique_count
    })

composite_df = pd.DataFrame(ck)

display(composite_df)

,table,key_columns,duplicate_combinations,missing_key_values,unique_combinations
0,driver_monthly_metrics,driver_id + month,0,0,4464
1,truck_utilization_metrics,truck_id + month,0,0,3312


In [6]:
# Validate the foreign key relationships

# Each item contains: child table, child column, parent table, parent column

relationships = [
    ("loads", "customer_id", "customers", "customer_id"),
    ("loads", "route_id", "routes", "route_id"),

    ("trips", "load_id", "loads", "load_id"),
    ("trips", "driver_id", "drivers", "driver_id"),
    ("trips", "truck_id", "trucks", "truck_id"),
    ("trips", "trailer_id", "trailers", "trailer_id"),

    ("delivery_events", "load_id", "loads", "load_id"),
    ("delivery_events", "trip_id", "trips", "trip_id"),
    ("delivery_events", "facility_id", "facilities", "facility_id"),

    ("fuel_purchases", "trip_id", "trips", "trip_id"),
    ("fuel_purchases", "driver_id", "drivers", "driver_id"),
    ("fuel_purchases", "truck_id", "trucks", "truck_id"),

    ("maintenance_records", "truck_id", "trucks", "truck_id"),

    ("safety_incidents", "trip_id", "trips", "trip_id"),
    ("safety_incidents", "driver_id", "drivers", "driver_id"),
    ("safety_incidents", "truck_id", "trucks", "truck_id"),

    ("driver_monthly_metrics", "driver_id", "drivers", "driver_id"),
    ("truck_utilization_metrics", "truck_id", "trucks", "truck_id")
]

fk = []

for child_table, child_column, parent_table, parent_column in relationships:

    child_df = csv_data[child_table]
    parent_df = csv_data[parent_table]

    # Remove empty values before checking whether IDs match
    child_ids = child_df[child_column].dropna()
    parent_ids = parent_df[parent_column].dropna()

    # Check which child IDs are not present in the parent table
    unmatched_ids = child_ids[~child_ids.isin(parent_ids)]

    fk.append({
        "child_table": child_table,
        "foreign_key": child_column,
        "parent_table": parent_table,
        "total_records": len(child_df),
        "missing_ids": child_df[child_column].isna().sum(),
        "unmatched_ids": len(unmatched_ids),
        "matched_ids": child_ids.isin(parent_ids).sum()
    })

fk_df = pd.DataFrame(fk)

display(fk_df)

,child_table,foreign_key,parent_table,total_records,missing_ids,unmatched_ids,matched_ids
0,loads,customer_id,customers,85410,0,0,85410
1,loads,route_id,routes,85410,0,0,85410
2,trips,load_id,loads,85410,0,0,85410
3,trips,driver_id,drivers,85410,1714,0,83696
4,trips,truck_id,trucks,85410,1672,0,83738
5,trips,trailer_id,trailers,85410,1680,0,83730
6,delivery_events,load_id,loads,170820,0,0,170820
7,delivery_events,trip_id,trips,170820,0,0,170820
8,delivery_events,facility_id,facilities,170820,0,0,170820
9,fuel_purchases,trip_id,trips,196442,0,0,196442


In [7]:
# Check missing values in key columns

key_columns = {
    "trips": ["driver_id", "truck_id", "trailer_id"],
    "fuel_purchases": ["driver_id", "truck_id", "trip_id"],
    "safety_incidents": ["driver_id", "truck_id", "trip_id"],
    "drivers": ["termination_date"]
}

missing_info = []

for table_name, columns in key_columns.items():

    df = csv_data[table_name]

    for column in columns:

        missing_count = df[column].isna().sum()
        missing_percentage = round(df[column].isna().mean() * 100, 2)

        missing_info.append({
            "table": table_name,
            "column": column,
            "total_records": len(df),
            "missing_records": missing_count,
            "missing_percentage": missing_percentage
        })

missing_df = pd.DataFrame(missing_info)

display(missing_df)

,table,column,total_records,missing_records,missing_percentage
0,trips,driver_id,85410,1714,2.01
1,trips,truck_id,85410,1672,1.96
2,trips,trailer_id,85410,1680,1.97
3,fuel_purchases,driver_id,196442,3988,2.03
4,fuel_purchases,truck_id,196442,3880,1.98
5,fuel_purchases,trip_id,196442,0,0.00
6,safety_incidents,driver_id,170,1,0.59
7,safety_incidents,truck_id,170,1,0.59
8,safety_incidents,trip_id,170,0,0.00
9,drivers,termination_date,150,124,82.67


In [8]:
# Check the date columns

date_columns = {
    "customers": ["contract_start_date"],
    "delivery_events": ["scheduled_datetime", "actual_datetime"],
    "drivers": ["date_of_birth", "hire_date", "termination_date"],
    "fuel_purchases": ["purchase_date"],
    "loads": ["load_date"],
    "maintenance_records": ["maintenance_date"],
    "safety_incidents": ["incident_date"],
    "trailers": ["acquisition_date"],
    "trips": ["dispatch_date"],
    "driver_monthly_metrics": ["month"],
    "truck_utilization_metrics": ["month"]
}

date_results = []

for table_name, columns in date_columns.items():

    df = csv_data[table_name]

    for column in columns:

        # Convert the column into a date format
        converted_dates = pd.to_datetime(df[column], errors="coerce")

        # Count values that were present but could not be converted
        invalid_count = (
            df[column].notna() & converted_dates.isna()
        ).sum()

        date_results.append({
            "table": table_name,
            "column": column,
            "invalid_dates": invalid_count,
            "missing_dates": df[column].isna().sum(),
            "earliest_date": converted_dates.min(),
            "latest_date": converted_dates.max()
        })

date_df = pd.DataFrame(date_results)

display(date_df)

,table,column,invalid_dates,missing_dates,earliest_date,latest_date
0,customers,contract_start_date,0,0,2020-01-07 00:00:00.000000,2022-01-01 00:00:00.000000
1,delivery_events,scheduled_datetime,0,0,2022-01-01 07:00:00.000000,2025-01-02 19:21:42.629833
2,delivery_events,actual_datetime,0,0,2022-01-01 06:13:28.892373,2025-01-03 00:29:47.403350
3,drivers,date_of_birth,0,0,1960-05-24 00:00:00.000000,1995-07-03 00:00:00.000000
4,drivers,hire_date,0,0,2012-01-28 00:00:00.000000,2021-12-06 00:00:00.000000
5,drivers,termination_date,0,124,2013-01-08 00:00:00.000000,2022-05-05 00:00:00.000000
6,fuel_purchases,purchase_date,0,0,2022-01-01 00:00:00.000000,2025-01-02 23:00:00.000000
7,loads,load_date,0,0,2022-01-01 00:00:00.000000,2024-12-31 00:00:00.000000
8,maintenance_records,maintenance_date,0,0,2022-01-01 00:00:00.000000,2024-12-31 00:00:00.000000
9,safety_incidents,incident_date,0,0,2022-01-05 02:00:00.000000,2024-12-24 09:00:00.000000


In [9]:
# Check the financial calculations

# Fuel purchase validation
fuel = csv_data["fuel_purchases"].copy()

# Calculate the fuel cost using gallons and price per gallon
fuel["calculated_cost"] = (
    fuel["gallons"] * fuel["price_per_gallon"]
)

# Compare the calculated cost with the recorded cost
fuel["cost_difference"] = (
    fuel["total_cost"] - fuel["calculated_cost"]
).round(2)

fuel_mismatches = fuel[
    fuel["cost_difference"].abs() > 0.02
]

print("FUEL PURCHASE VALIDATION")
print("Total records:", len(fuel))
print("Mismatched calculations:", len(fuel_mismatches))




# Maintenance cost validation
maintenance = csv_data["maintenance_records"].copy()

# Add labour cost and parts cost to get the calculated total
maintenance["calculated_cost"] = (
    maintenance["labor_cost"] + maintenance["parts_cost"]
)

maintenance["cost_difference"] = (
    maintenance["total_cost"] - maintenance["calculated_cost"]
).round(2)

maintenance_mismatches = maintenance[
    maintenance["cost_difference"].abs() > 0.02
]

print("\nMAINTENANCE VALIDATION")
print("Total records:", len(maintenance))
print("Mismatched calculations:", len(maintenance_mismatches))



FUEL PURCHASE VALIDATION
Total records: 196442
Mismatched calculations: 0

MAINTENANCE VALIDATION
Total records: 2920
Mismatched calculations: 0


In [10]:
# Check delivery punctuality

events = csv_data["delivery_events"].copy()

# Convert both datetime columns into proper datetime values
events["scheduled_datetime"] = pd.to_datetime(
    events["scheduled_datetime"], errors="coerce"
)

events["actual_datetime"] = pd.to_datetime(
    events["actual_datetime"], errors="coerce"
)

# Calculate the delay in minutes
events["delay_minutes"] = (
    events["actual_datetime"] - events["scheduled_datetime"]
).dt.total_seconds() / 60

# For this check, a delivery is on time if it is not late
events["calculated_on_time"] = events["delay_minutes"] <= 0

# Compare the calculated flag with the flag already in the dataset
events["flag_mismatch"] = (
    events["on_time_flag"] != events["calculated_on_time"]
)

print("DELIVERY PUNCTUALITY VALIDATION")
print("Total events:", len(events))

invalid_timestamps = (
    events["scheduled_datetime"].isna().sum()
    + events["actual_datetime"].isna().sum()
)

print("Missing or invalid timestamps:", invalid_timestamps)
print("On-time flag mismatches:", events["flag_mismatch"].sum())
print("Average delay in minutes:", round(events["delay_minutes"].mean(), 2))

# Display few records where the two flags do not match
display(events.loc[
    events["flag_mismatch"],
    [
        "event_id",
        "event_type",
        "scheduled_datetime",
        "actual_datetime",
        "delay_minutes",
        "on_time_flag",
        "calculated_on_time"
    ]
].head(20))

DELIVERY PUNCTUALITY VALIDATION
Total events: 170820
Missing or invalid timestamps: 0
On-time flag mismatches: 56866
Average delay in minutes: 75.01


,event_id,event_type,scheduled_datetime,actual_datetime,delay_minutes,on_time_flag,calculated_on_time
10,EVT00000011,Pickup,2022-01-01 10:00:00.000000,2022-01-01 11:11:18.602953,71.310049,True,False
11,EVT00000012,Delivery,2022-01-03 02:35:18.602953,2022-01-02 23:52:34.261273,-162.739028,False,True
23,EVT00000024,Delivery,2022-01-02 08:54:16.218829,2022-01-02 09:00:21.031626,6.080213,True,False
28,EVT00000029,Pickup,2022-01-01 07:00:00.000000,2022-01-01 07:47:13.542430,47.225707,True,False
33,EVT00000034,Delivery,2022-01-02 16:12:02.952632,2022-01-02 17:29:23.553207,77.343343,True,False
35,EVT00000036,Delivery,2022-01-02 13:30:26.571604,2022-01-02 11:18:22.416523,-132.069251,False,True
38,EVT00000039,Pickup,2022-01-01 14:00:00.000000,2022-01-01 14:18:54.697099,18.911618,True,False
40,EVT00000041,Pickup,2022-01-01 10:00:00.000000,2022-01-01 11:44:40.281617,104.671360,True,False
42,EVT00000043,Pickup,2022-01-01 07:00:00.000000,2022-01-01 08:40:22.808653,100.380144,True,False
44,EVT00000045,Pickup,2022-01-01 12:00:00.000000,2022-01-01 12:42:31.373576,42.522893,True,False


In [11]:
# summary of all validation checks

summary_validation = pd.DataFrame([
    {
        "validation": "Primary Key",
        "records_checked": len(pk_df),
        "issues_found": (
            pk_df["duplicate_keys"].sum()
            + pk_df["missing_keys"].sum()
        )
    },
    {
        "validation": "Composite Key",
        "records_checked": len(composite_df),
        "issues_found": (
            composite_df["duplicate_combinations"].sum()
            + composite_df["missing_key_values"].sum()
        )
    },
    {
        "validation": "Foreign Key",
        "records_checked": len(fk_df),
        "issues_found": (
            fk_df["unmatched_ids"].sum()
            + fk_df["missing_ids"].sum()
        )
    },
    {
        "validation": "Important Missing Values",
        "records_checked": len(missing_df),
        "issues_found": missing_df["missing_records"].sum()
    },
    {
        "validation": "Date Validation",
        "records_checked": len(date_df),
        "issues_found": date_df["invalid_dates"].sum()
    },
    {
        "validation": "Fuel Calculation",
        "records_checked": len(fuel),
        "issues_found": len(fuel_mismatches)
    },
    {
        "validation": "Maintenance Calculation",
        "records_checked": len(maintenance),
        "issues_found": len(maintenance_mismatches)
    },
    {
        "validation": "Delivery Punctuality",
        "records_checked": len(events),
        "issues_found": events["flag_mismatch"].sum()
    }
])

display(summary_validation)

,validation,records_checked,issues_found
0,Primary Key,12,0
1,Composite Key,2,0
2,Foreign Key,18,12936
3,Important Missing Values,10,13060
4,Date Validation,14,0
5,Fuel Calculation,196442,0
6,Maintenance Calculation,2920,0
7,Delivery Punctuality,170820,56866


In [12]:
# Save the validation reports

# Create a folder to store the output files
output_directory = Path(
    r"C:\Users\manoj\OneDrive\Documents\logistic operation database\validation reports"
)

output_directory.mkdir(parents=True, exist_ok=True)

# Save each validation result 
pk_df.to_csv(output_directory / "primary_key_validation.csv", index=False)

composite_df.to_csv(
    output_directory / "composite_key_validation.csv", index=False
)

fk_df.to_csv(output_directory / "foreign_key_validation.csv", index=False)

missing_df.to_csv(
    output_directory / "missing_value_validation.csv", index=False
)

date_df.to_csv(output_directory / "date_validation.csv", index=False)

summary_validation.to_csv(
    output_directory / "validation_summary.csv", index=False
)

print("Validation reports saved successfully.")
print("Saved location:", output_directory)

Validation reports saved successfully.
Saved location: C:\Users\manoj\OneDrive\Documents\logistic operation database\validation reports


In [13]:
# Investigate delivery flag mismatches by event type

delivery_investigation = (
    events.groupby("event_type")
    .agg(
        total_events=("event_id", "count"),
        on_time_flag_mismatches=("flag_mismatch", "sum"),
        average_delay_minutes=("delay_minutes", "mean"),
        actual_early_events=("delay_minutes", lambda values: (values < 0).sum()),
        actual_late_events=("delay_minutes", lambda values: (values > 0).sum()),
        actual_on_schedule=("delay_minutes", lambda values: (values == 0).sum())
    )
    .reset_index()
)

# Calculate the percentage of mismatches for each event type
delivery_investigation["mismatch_percentage"] = (
    delivery_investigation["on_time_flag_mismatches"]
    / delivery_investigation["total_events"] * 100
).round(2)

display(delivery_investigation)

,event_type,total_events,on_time_flag_mismatches,average_delay_minutes,actual_early_events,actual_late_events,actual_on_schedule,mismatch_percentage
0,Delivery,85410,28444,90.315741,28422,56988,0,33.30
1,Pickup,85410,28422,59.714077,28571,56839,0,33.28


In [14]:
# Compare the original and calculated on-time flags

flag_compare = pd.crosstab(
    events["on_time_flag"],
    events["calculated_on_time"],
    rownames=["Original Flag"],
    colnames=["Calculated Flag"]
)

display(flag_compare)

Calculated Flag,False,True
Original Flag,,
False,66343,9382
True,47484,47611


In [15]:
# Check the delay distribution for each original on-time flag

summary_delay = (
    events.groupby("on_time_flag")["delay_minutes"]
    .agg(
        total_events="count",
        average_delay="mean",
        minimum_delay="min",
        maximum_delay="max",
        median_delay="median"
    )
    .round(2)
)

display(summary_delay)

,total_events,average_delay,minimum_delay,maximum_delay,median_delay
on_time_flag,,,,,
False,75725,169.30,-179.99,360.0,192.19
True,95095,-0.07,-120.00,120.0,-0.17


In [16]:
# Test different on-time tolerance windows

tolerance_info = []

# Check how the results change with different tolerance limits
for tolerance in [0, 15, 30, 60, 90, 120, 150, 180]:

    # Treat deliveries within the selected early/late window as on time
    calculated_flag = events["delay_minutes"].between(
        -tolerance, tolerance
    )

    # Compare the new flag with the original flag
    mismatches = (
        calculated_flag != events["on_time_flag"]
    ).sum()

    mismatch_percentage = round(
        mismatches / len(events) * 100, 2
    )

    tolerance_info.append({
        "tolerance_minutes": tolerance,
        "mismatches": mismatches,
        "mismatch_percentage": mismatch_percentage
    })

tolerance_df = pd.DataFrame(tolerance_info)

display(tolerance_df)

,tolerance_minutes,mismatches,mismatch_percentage
0,0,95095,55.67
1,15,83228,48.72
2,30,71067,41.60
3,60,47295,27.69
4,90,23741,13.90
5,120,0,0.00
6,150,16597,9.72
7,180,32979,19.31
